- In this notebook: improved flare duration function for reference

In [3]:
#required packages for duration function only
import numpy as np

import datetime as dt
from sunpy.time import parse_time

import pandas as pd

In [4]:
#function takes a list of flare dataframes (in 25-50 keV and 50-100 keV) and their corresponding flare list indices
# - this is so the flare list can provide a starting point of a temporal region to search for the HXR flare signal
def flare_durations(list_of_flare_dfs, flare_list_locations):

    #storing duration information
    durations = []
    estimated_starts = []
    estimated_ends = []
    durations_50100 = []
    estimated_starts_50100 = []
    estimated_ends_50100 = []
    
    for ind, df in enumerate(list_of_flare_dfs):
        #taking 20 minutes either side of flare list times (HXR is sometimes not well contained by these)
        start = parse_time(flarelist['start_UTC'].loc[flare_list_locations[ind]]).datetime - dt.timedelta(minutes=20)
        end = parse_time(flarelist['end_UTC'].loc[flare_list_locations[ind]]).datetime  + dt.timedelta(minutes=20)
    
        flare_df = df.truncate(start, end)
        flare_df.index = parse_time(flare_df.index).datetime 

        peak = np.max(flare_df['25.0 keV-50.0 keV'])
        peak_loc = flare_df.index[flare_df['25.0 keV-50.0 keV'] == peak][0]

        #start is found by taking a point where the increase from the previous data point is >0.2 counts, and then
        #checking that the next 20 points (for 1s resampled data) are monotonically increasing
        start_df = flare_df[(flare_df['25.0 keV-50.0 keV'].diff()>0.2) & (flare_df.index<peak_loc)]

        found_start=False
        for time in start_df.index:
            start = flare_df.index.get_loc(flare_df.index[flare_df.index == time][0])
            points = flare_df['25.0 keV-50.0 keV'].iloc[start:start+20]
            if points.is_monotonic_increasing:
                estimated_start = time
                found_start=True
                break
        #very few flares will not have a start registed this way -  usually only SEP contaminated, but will return
        #start is nan if so
        if not found_start:
            estimated_start=np.nan

        #flare end is found from where the counts go below the mean (of the whole dataframe - not just the flaring region)
        end_df = flare_df[(flare_df['25.0 keV-50.0 keV']<np.mean(df['25.0 keV-50.0 keV'])) & (flare_df.index>peak_loc)]
        if not end_df.empty:
            estimated_end = end_df.index[0]
        #similarly returns nan if no end found
        else:
            estimated_end = np.nan
    
        estimated_starts.append(estimated_start)
        estimated_ends.append(estimated_end)
        
        #also returns flare duration in seconds
        if (pd.notna(estimated_start)) & (pd.notna(estimated_end)):
            estimated_duration = (estimated_end - estimated_start).total_seconds()
            durations.append(estimated_duration)
        else:
            durations.append(np.nan)

        #process is repeated for 50-100 keV with slightly different criteria
        #- only if both start & end in 25-50 have been found

        if (pd.notna(estimated_start)) & (pd.notna(estimated_end)):
            flare_df_50100 = flare_df.truncate(estimated_start, estimated_end)
    
            peak_50100 = np.max(flare_df_50100['50.0 keV-100.0 keV'])
            peak_loc_50100 = flare_df_50100.index[flare_df_50100['50.0 keV-100.0 keV'] == peak_50100][0]
    
            start_df_50100 = flare_df_50100[(flare_df_50100['50.0 keV-100.0 keV'].diff()>0.1) & (flare_df_50100.index<peak_loc_50100)]
    
            found_start_50100=False
            for time in start_df_50100.index:
                start = flare_df_50100.index.get_loc(flare_df_50100.index[flare_df_50100.index == time][0])
                points = flare_df_50100['50.0 keV-100.0 keV'].iloc[start:start+5]
                if points.is_monotonic_increasing:
                    estimated_start_50100 = time
                    found_start_50100=True
                    break
    
            if not found_start_50100:
                estimated_start_50100=np.nan
    
            end_df_50100 = flare_df_50100[(flare_df_50100['50.0 keV-100.0 keV']<2*np.mean(df['50.0 keV-100.0 keV'])) & (flare_df_50100.index>peak_loc_50100)]
            if not end_df_50100.empty:
                estimated_end_50100 = end_df_50100.index[0]
            else:
                estimated_end_50100 = np.nan
        
            estimated_starts_50100.append(estimated_start_50100)
            estimated_ends_50100.append(estimated_end_50100)
            
            if (pd.notna(estimated_start_50100)) & (pd.notna(estimated_end_50100)):
                estimated_duration_50100 = (estimated_end_50100 - estimated_start_50100).total_seconds()
                durations_50100.append(estimated_duration_50100)
            else:
                durations_50100.append(np.nan)
                
        else:
            durations_50100.append(np.nan)
            estimated_starts_50100.append(np.nan)
            estimated_ends_50100.append(np.nan)

    times_25_50 = [estimated_starts, estimated_ends, durations]
    times_50_100 = [estimated_starts_50100, estimated_ends_50100, durations_50100]
            
    return times_25_50, times_50_100